<a href="https://colab.research.google.com/github/JeevanS-0721/Employee-Wellness-Management-/blob/main/NLP_Preprocessing_Pipeline.ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



| Secret name | Value |
|---|---|
| `DB_HOST` | e.g. `ep-xxxx.us-east-2.aws.neon.tech` (from Neon/Supabase) |
| `DB_PORT` | usually `5432` |
| `DB_NAME` | your database name |
| `DB_USER` | your database user |
| `DB_PASSWORD` | your database password |
| `JWT_SECRET` | any long random string (generate one in the next cell) |
| `SMTP_EMAIL` | your Gmail address |
| `SMTP_APP_PASSWORD` | 16-character Gmail **App Password** (not your real password) |
| `NGROK_AUTHTOKEN` | from https://dashboard.ngrok.com/get-started/your-authtoken |

**Note:** the FastAPI backend added below reuses `JWT_SECRET` — no additional secrets are needed for it.


In [ ]:
!pip install -q streamlit psycopg2-binary PyJWT bcrypt \
    python-dotenv email-validator pyngrok \
    fastapi uvicorn python-multipart requests \
    langdetect ftfy emoji deep-translator vaderSentiment spacy pandas matplotlib \
    transformers accelerate torch stopwordsiso
!python -m spacy download xx_sent_ud_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 62.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('xx_sent_ud_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


## New: Employee Wellness NLP Analysis

After login, the app now has an **"Run NLP Analysis"** button next to the file upload. It sends the CSV/TXT to a new `/analyze` endpoint on the FastAPI backend, which detects language (Telugu/Kannada-aware), cleans and tokenizes the text, translates it to English, lemmatizes, and runs VADER sentiment + keyword-based emotion detection. Results (including sentiment/emotion bar charts) render inline in Streamlit.

**No new secrets needed** — this reuses `JWT_SECRET` like the upload feature already did.

**Heads-up:** installing spaCy + downloading the `xx_sent_ud_sm` model adds a few minutes to Section 2's install cell the first time you run it in a fresh Colab runtime.


In [ ]:
from google.colab import userdata

required_secrets = [
    "DB_HOST", "DB_PORT", "DB_NAME", "DB_USER", "DB_PASSWORD",
    "JWT_SECRET", "SMTP_EMAIL", "SMTP_APP_PASSWORD", "NGROK_AUTHTOKEN",
]

values = {}
missing = []
for key in required_secrets:
    try:
        values[key] = userdata.get(key)
    except Exception:
        missing.append(key)

if missing:
    raise RuntimeError(
        f"Missing Colab secrets: {missing}. "
        f"Add them via the key icon in the left sidebar, then re-run this cell."
    )

env_content = f'''DB_HOST={values["DB_HOST"]}
DB_PORT={values["DB_PORT"]}
DB_NAME={values["DB_NAME"]}
DB_USER={values["DB_USER"]}
DB_PASSWORD={values["DB_PASSWORD"]}

JWT_SECRET={values["JWT_SECRET"]}
JWT_ALGORITHM=HS256
JWT_EXPIRY_MINUTES=60

SMTP_HOST=smtp.gmail.com
SMTP_PORT=587
SMTP_EMAIL={values["SMTP_EMAIL"]}
SMTP_APP_PASSWORD={values["SMTP_APP_PASSWORD"]}

OTP_EXPIRY_MINUTES=10
'''

with open(".env", "w") as f:
    f.write(env_content)

print("Wrote .env with", len(values), "secrets loaded.")

Wrote .env with 9 secrets loaded.


In [ ]:
%%writefile db.py
import os, psycopg2
from psycopg2.extras import RealDictCursor
from contextlib import contextmanager
from dotenv import load_dotenv
load_dotenv()

CFG = dict(host=os.getenv("DB_HOST"), port=os.getenv("DB_PORT", "5432"),
           dbname=os.getenv("DB_NAME"), user=os.getenv("DB_USER"),
           password=os.getenv("DB_PASSWORD"), sslmode="require")

@contextmanager
def cursor(commit=False):
    conn = psycopg2.connect(**CFG)
    cur = conn.cursor(cursor_factory=RealDictCursor)
    try:
        yield cur
        if commit: conn.commit()
    finally:
        cur.close(); conn.close()

def init_db():
    with cursor(commit=True) as cur:
        cur.execute("""CREATE TABLE IF NOT EXISTS users (
            id SERIAL PRIMARY KEY, username VARCHAR(50) UNIQUE, email VARCHAR(255) UNIQUE,
            password_hash VARCHAR(255), is_verified BOOLEAN DEFAULT FALSE)""")
        cur.execute("""CREATE TABLE IF NOT EXISTS otp_codes (
            id SERIAL PRIMARY KEY, email VARCHAR(255), code VARCHAR(6),
            purpose VARCHAR(20), expires_at TIMESTAMP, used BOOLEAN DEFAULT FALSE)""")

Writing db.py


In [ ]:
%%writefile auth.py
import os, jwt, bcrypt, random, string
from datetime import datetime, timedelta, timezone
from dotenv import load_dotenv
from db import cursor
load_dotenv()

SECRET = os.getenv("JWT_SECRET")

def hash_pw(pw): return bcrypt.hashpw(pw.encode(), bcrypt.gensalt()).decode()
def check_pw(pw, h): return bcrypt.checkpw(pw.encode(), h.encode())

def make_token(user):
    payload = {"id": user["id"], "username": user["username"], "email": user["email"],
               "exp": datetime.now(timezone.utc) + timedelta(hours=1)}
    return jwt.encode(payload, SECRET, algorithm="HS256")

def read_token(token):
    try: return jwt.decode(token, SECRET, algorithms=["HS256"])
    except jwt.PyJWTError: return None

def get_user(email):
    with cursor() as cur:
        cur.execute("SELECT * FROM users WHERE email=%s", (email,))
        return cur.fetchone()

def username_taken(username):
    with cursor() as cur:
        cur.execute("SELECT 1 FROM users WHERE username=%s", (username,))
        return cur.fetchone() is not None

def create_user(username, email, pw):
    with cursor(commit=True) as cur:
        cur.execute("INSERT INTO users (username,email,password_hash) VALUES (%s,%s,%s)",
                    (username, email, hash_pw(pw)))

def verify_user(email):
    with cursor(commit=True) as cur:
        cur.execute("UPDATE users SET is_verified=TRUE WHERE email=%s", (email,))

def set_password(email, pw):
    with cursor(commit=True) as cur:
        cur.execute("UPDATE users SET password_hash=%s WHERE email=%s", (hash_pw(pw), email))

def new_otp():
    return "".join(random.choices(string.digits, k=6))

def save_otp(email, code, purpose):
    exp = datetime.now(timezone.utc) + timedelta(minutes=10)
    with cursor(commit=True) as cur:
        cur.execute("UPDATE otp_codes SET used=TRUE WHERE email=%s AND purpose=%s", (email, purpose))
        cur.execute("INSERT INTO otp_codes (email,code,purpose,expires_at) VALUES (%s,%s,%s,%s)",
                    (email, code, purpose, exp))

def check_otp(email, code, purpose):
    with cursor(commit=True) as cur:
        cur.execute("""SELECT * FROM otp_codes WHERE email=%s AND purpose=%s AND used=FALSE
                       ORDER BY id DESC LIMIT 1""", (email, purpose))
        row = cur.fetchone()
        if not row or row["code"] != code:
            return False
        now = datetime.now(row["expires_at"].tzinfo) if row["expires_at"].tzinfo else datetime.now()
        if now > row["expires_at"]:
            return False
        cur.execute("UPDATE otp_codes SET used=TRUE WHERE id=%s", (row["id"],))
        return True

Writing auth.py


In [ ]:
%%writefile email_utils.py
import os, smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from dotenv import load_dotenv
load_dotenv()

HOST, PORT = "smtp.gmail.com", 587
EMAIL = os.getenv("SMTP_EMAIL")
APP_PW = os.getenv("SMTP_APP_PASSWORD")

def send_otp(to_email, code, purpose):
    # Set up subject line and heading text based on the purpose
    if purpose == "signup":
        subject = "Welcome to Mood Mentor! Verify your email"
        heading = "Welcome to Mood Mentor!"
        subheading = "Thank you for joining us. Please use the verification code below to complete your registration."
    else:
        subject = "Reset your Mood Mentor Password"
        heading = "Password Reset Request"
        subheading = "We received a request to reset your password. Use the verification code below to proceed."

    # HTML content for a beautifully styled email
    html_content = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="utf-8">
        <style>
            body {{
                font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
                background-color: #f4f6f9;
                margin: 0;
                padding: 0;
            }}
            .email-container {{
                max-width: 600px;
                margin: 40px auto;
                background-color: #ffffff;
                border-radius: 8px;
                overflow: hidden;
                box-shadow: 0 4px 10px rgba(0, 0, 0, 0.05);
                border: 1px solid #e1e8ed;
            }}
            .header {{
                background-color: #4F46E5; /* Royal indigo color */
                padding: 30px 20px;
                text-align: center;
                color: #ffffff;
            }}
            .header h1 {{
                margin: 0;
                font-size: 24px;
                font-weight: 600;
            }}
            .content {{
                padding: 40px 30px;
                color: #333333;
                line-height: 1.6;
            }}
            .content p {{
                margin: 0 0 20px 0;
                font-size: 16px;
            }}
            .otp-box {{
                background-color: #F3F4F6;
                border-radius: 6px;
                padding: 15px;
                text-align: center;
                font-size: 32px;
                font-weight: bold;
                letter-spacing: 5px;
                color: #1F2937;
                margin: 30px 0;
                border: 1px dashed #D1D5DB;
            }}
            .footer {{
                background-color: #F9FAFB;
                padding: 20px;
                text-align: center;
                font-size: 12px;
                color: #9CA3AF;
                border-top: 1px solid #E5E7EB;
            }}
        </style>
    </head>
    <body>
        <div class="email-container">
            <div class="header">
                <h1>{heading}</h1>
            </div>
            <div class="content">
                <p>Hello,</p>
                <p>{subheading}</p>

                <div class="otp-box">
                    {code}
                </div>

                <p>This verification code is valid for <strong>10 minutes</strong>. If you did not make this request, you can safely ignore this email.</p>
                <p>Warmly,<br>The Mood Mentor Team</p>
            </div>
            <div class="footer">
                <p>Sent with care by Mood Mentor • Employee Wellness Analysis</p>
            </div>
        </div>
    </body>
    </html>
    """

    # Using MIMEMultipart to send an HTML payload safely
    msg = MIMEMultipart("alternative")
    msg["From"] = EMAIL
    msg["To"] = to_email
    msg["Subject"] = subject

    # Fallback plain text for old mail clients
    plain_text = f"Hello,\n\nYour verification code is: {code}\n\nThis code expires in 10 minutes."

    msg.attach(MIMEText(plain_text, "plain"))
    msg.attach(MIMEText(html_content, "html"))

    try:
        with smtplib.SMTP(HOST, PORT, timeout=15) as s:
            s.starttls()
            s.login(EMAIL, APP_PW)
            s.sendmail(EMAIL, to_email, msg.as_string())
        return True, "sent"
    except Exception as e:
        return False, str(e)

Writing email_utils.py


In [ ]:
%%writefile app.py
import os, re, requests, streamlit as st
from db import init_db
from auth import (make_token, read_token, get_user, username_taken, create_user,
                  verify_user, set_password, check_pw, new_otp, save_otp, check_otp)
from email_utils import send_otp

# 1. Page Configuration
st.set_page_config(page_title="MOODMENTOR", page_icon="😊", layout="wide")
BACKEND_URL = os.getenv("BACKEND_URL", "http://localhost:8000")

@st.cache_resource
def setup(): init_db()
setup()

# 2. Enforce Light Shade of Green Theme with High Contrast Black Text
st.markdown("""
    <style>
    /* --- GRADIENT BACKGROUND --- */
    .stApp, [data-testid="stAppViewContainer"] {
        background: linear-gradient(135deg, #F0FDF4 0%, #D1FAE5 100%) !important;
        color: #0F172A !important;
    }

    /* --- REMOVE BLACK HIGHLIGHTS & FIX FILE UPLOADER --- */
    header[data-testid="stHeader"] { background: transparent !important; }
    [data-testid="stBottom"], [data-testid="stBottomBlockContainer"] { background: transparent !important; }

    /* Fix Chat Input Visibility */
    [data-testid="stChatInput"] { background-color: transparent !important; }
    [data-testid="stChatInput"] > div {
        background-color: #FFFFFF !important;
        border: 2px solid #86EFAC !important;
        border-radius: 8px !important;
        box-shadow: 0 4px 6px rgba(0, 0, 0, 0.05) !important;
    }
    [data-testid="stChatInput"] textarea { color: #0F172A !important; }
    [data-testid="stChatInput"] textarea::placeholder { color: #64748B !important; }
    [data-testid="stChatInput"] button {
        background-color: transparent !important;
        color: #166534 !important;
        box-shadow: none !important;
    }
    [data-testid="stChatInput"] button:hover { background-color: #F0FDF4 !important; }

    /* Fix File Uploader Dropzone and Button */
    [data-testid="stFileUploader"] { background-color: transparent !important; }
    [data-testid="stFileUploaderDropzone"] {
        background-color: #FFFFFF !important;
        border: 2px dashed #166534 !important;
        border-radius: 8px !important;
    }
    [data-testid="stFileUploaderDropzone"] * { color: #0F172A !important; }
    [data-testid="stFileUploaderDropzone"] button {
        background-color: #FFFFFF !important;
        color: #0F172A !important;
        border: 1px solid #166534 !important;
    }
    [data-testid="stFileUploaderDropzone"] button:hover {
        background-color: #F0FDF4 !important;
        color: #166534 !important;
    }

    /* --- GENERAL TYPOGRAPHY & COLORS --- */
    p, span, li, label, div { color: #0F172A !important; }
    h1, h2, h3, [data-testid="stMarkdownContainer"] h1, [data-testid="stMarkdownContainer"] h2, [data-testid="stMarkdownContainer"] h3 {
        color: #166534 !important;
        font-weight: 700 !important;
    }
    [data-testid="stWidgetLabel"] p, label p, .stSlider label {
        color: #0F172A !important;
        font-weight: 600 !important;
    }

    /* --- SIDEBAR --- */
    [data-testid="stSidebar"] {
        background-color: #FFFFFF !important;
        border-right: 1px solid #BBF7D0 !important;
    }

    /* --- INPUTS & DROPDOWNS --- */
    input, textarea, [data-baseweb="input"], [data-baseweb="select"] {
        background-color: #FFFFFF !important;
        color: #0F172A !important;
        border: 1px solid #86EFAC !important;
        border-radius: 6px !important;
    }
    input { -webkit-text-fill-color: #0F172A !important; }
    [data-testid="stSidebar"] div[data-baseweb="select"] > div {
        background-color: #FFFFFF !important;
        color: #0F172A !important;
        border: 1px solid #86EFAC !important;
    }

    div[data-baseweb="popover"], div[data-baseweb="popover"] > div, div[role="listbox"], ul[data-baseweb="menu"] {
        background-color: #FFFFFF !important;
    }
    li[role="option"] { background-color: #FFFFFF !important; color: #0F172A !important; }
    li[role="option"] span { color: #0F172A !important; -webkit-text-fill-color: #0F172A !important; }
    li[role="option"]:hover, li[role="option"]:hover span {
        background-color: #F0FDF4 !important;
        color: #166534 !important;
        -webkit-text-fill-color: #166534 !important;
    }

    /* --- BUTTONS --- */
    div.stButton > button {
        background-color: #2563EB !important;
        color: #FFFFFF !important;
        border: none !important;
        border-radius: 6px !important;
        font-weight: 600 !important;
        padding: 0.5rem 1rem !important;
        box-shadow: 0 2px 4px rgba(0,0,0,0.05) !important;
    }
    div.stButton > button:hover { background-color: #1D4ED8 !important; color: #FFFFFF !important; }
    </style>
""", unsafe_allow_html=True)

# Session State Persistence
if "token" not in st.session_state: st.session_state.token = None
if "chat_history" not in st.session_state: st.session_state.chat_history = []
if "signup_email_target" not in st.session_state: st.session_state.signup_email_target = ""
if "forgot_target_email" not in st.session_state: st.session_state.forgot_target_email = ""

def valid_pw(pw): return len(pw) >= 8 and re.search(r"[A-Za-z]", pw) and re.search(r"[0-9]", pw)

# Cleaned up analysis output function
def display_analysis_results(r):
    st.success("✅ Analysis Complete")

    st.write("**Analyzed Text:**", r.get("raw_text", "N/A"))
    st.divider()

    st.subheader("Analysis Results")
    scores = r["sentiment_scores"]
    c1, c2 = st.columns(2)
    with c1:
        st.write(f"**Sentiment:** {r['final_sentiment']}")
        st.bar_chart({"Positive": scores["pos"], "Negative": scores["neg"], "Neutral": scores["neu"]})
    with c2:
        st.write(f"**Dominant Emotion:** {r['final_emotion']}")
        st.bar_chart(r["emotion_scores"])

# Navigation Manager
if st.session_state.token:
    menu_options = ["📊 Wellness Analysis", "🚪 Logout"]
else:
    menu_options = ["🏠 Home", "📝 Sign Up", "🔐 Login", "🔄 Forgot Password"]

choice = st.sidebar.selectbox("Navigation Menu", menu_options)

# ---------------- 1. HOME SCREEN ----------------
if choice == "🏠 Home":
    st.title("😊 MOODMENTOR")
    st.write("Welcome to MOODMENTOR. Monitor, analyze, and map employee wellness trends cleanly.")
    st.info("💡 Use the sidebar menu to register or access your secure portal.")

# ---------------- 2. SIGN UP WITH VALIDATION & OTP ----------------
elif choice == "📝 Sign Up":
    st.title("📝 Create Account")
    reg_username = st.text_input("Username")
    reg_email = st.text_input("Email Address")
    reg_pass = st.text_input("Password", type="password")
    reg_confirm = st.text_input("Confirm Password", type="password")
    email_regex = r"^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+$"

    if st.button("Send Verification OTP"):
        if not reg_username or not reg_email or not reg_pass: st.error("All registration input values must be filled out.")
        elif reg_pass != reg_confirm: st.error("Passwords do not match.")
        elif not valid_pw(reg_pass): st.error("Password needs 8+ chars, letters and numbers.")
        elif not re.match(email_regex, reg_email): st.error("Invalid email address format.")
        elif get_user(reg_email.strip().lower()): st.error("This email is already registered.")
        elif username_taken(reg_username): st.error("This username is already taken.")
        else:
            email_clean = reg_email.strip().lower()
            create_user(reg_username, email_clean, reg_pass)
            code = new_otp(); save_otp(email_clean, code, "signup")
            ok, msg = send_otp(email_clean, code, "signup")
            if ok:
                st.session_state.signup_email_target = email_clean
                st.success(f"Verification code sent successfully to {email_clean}!")
            else: st.error(f"SMTP Transmission Failed: {msg}")

    st.divider()
    input_signup_otp = st.text_input("Enter 6-Digit Email OTP", max_chars=6)
    if st.button("Verify OTP & Activate Profile"):
        if not st.session_state.signup_email_target: st.error("Please request a verification token first.")
        else:
            if check_otp(st.session_state.signup_email_target, input_signup_otp.strip(), "signup"):
                verify_user(st.session_state.signup_email_target)
                st.session_state.signup_email_target = ""
                st.success("Account verified successfully! Switch to the Login menu.")
            else: st.error("Incorrect or expired verification code.")

# ---------------- 3. LOGIN SCREEN ----------------
elif choice == "🔐 Login":
    st.title("🔐 User Authentication")
    login_email = st.text_input("Email Address")
    login_password = st.text_input("Password", type="password")

    if st.button("Authenticate"):
        user_record = get_user(login_email.strip().lower())
        if not user_record or not check_pw(login_password, user_record["password_hash"]):
            st.error("Access Denied: Invalid email address or account password.")
        elif not user_record["is_verified"]:
            st.warning("Account not verified. Please complete signup verification.")
        else:
            st.session_state.token = make_token(user_record)
            st.success("Access Granted! Cryptographic user session authenticated.")
            st.rerun()

# ---------------- 4. FORGOT PASSWORD SCREEN ----------------
elif choice == "🔄 Forgot Password":
    st.title("🔄 Reset Account Credentials")
    target_email = st.text_input("Enter Registered Email Address")
    if st.button("Send Reset Token Code"):
        email_clean = target_email.strip().lower()
        if get_user(email_clean):
            code = new_otp(); save_otp(email_clean, code, "password_reset")
            ok, msg = send_otp(email_clean, code, "password_reset")
            if ok:
                st.session_state.forgot_target_email = email_clean
                st.success(f"Security recovery code sent safely to {email_clean}.")
            else: st.error(f"SMTP Gateway transmission dropped: {msg}")
        else: st.error("This email address is not registered in our records.")

    st.divider()
    input_forgot_otp = st.text_input("Enter 6-Digit Recovery OTP", max_chars=6)
    new_password = st.text_input("New Password String", type="password")
    confirm_new_password = st.text_input("Confirm New Password String", type="password")

    if st.button("Update Database Encryption Profile"):
        if not st.session_state.forgot_target_email: st.error("Please initiate a recovery pass request first.")
        elif new_password != confirm_new_password: st.error("Passwords do not match.")
        elif not valid_pw(new_password): st.error("Password needs 8+ chars, letters and numbers.")
        else:
            if check_otp(st.session_state.forgot_target_email, input_forgot_otp.strip(), "password_reset"):
                set_password(st.session_state.forgot_target_email, new_password)
                st.session_state.forgot_target_email = ""
                st.success("Password update complete! You can now navigate back to Login.")
            else: st.error("Token verification match failed or expired.")

# ---------------- 5. AUTHENTICATED DASHBOARD (NLP & CHAT) ----------------
elif choice == "📊 Wellness Analysis":
    user = read_token(st.session_state.token)
    if user:
        st.title("📊 Employee Wellness Analytics Dashboard")
        st.caption(f"Secure Cryptographic Session Active: {user['username']} ({user['email']})")

        # 3-column layout: Upload (wide) | Divider (narrow) | Chat (wide)
        upload_col, divider_col, chat_col = st.columns([1.5, 0.1, 1])

        # NLP Pipeline Input UI
        with upload_col:
            headers = {"Authorization": f"Bearer {st.session_state.token}"}

            st.write("#### ✍️ Direct Text Input")
            direct_text = st.text_area("Enter feedback text:", height=100)
            if st.button("Analyze Text"):
                if direct_text:
                    with st.spinner("Processing pipeline..."):
                        resp = requests.post(f"{BACKEND_URL}/analyze", data={"text_input": direct_text}, headers=headers)
                        if resp.status_code == 200: display_analysis_results(resp.json())
                        else: st.error("Analysis failed.")
                else: st.warning("Please enter text first.")

            st.divider()

            st.write("#### 📁 File Upload")
            uploaded = st.file_uploader("Upload a CSV or TXT file", type=["csv", "txt"])
            if uploaded:
                column_name = st.text_input("Feedback column name (CSV only, leave blank for last column)") if uploaded.name.endswith(".csv") else None
                if st.button("Analyze File"):
                    with st.spinner("Processing pipeline..."):
                        resp = requests.post(f"{BACKEND_URL}/analyze",
                                             files={"file": (uploaded.name, uploaded.getvalue())},
                                             data={"column": column_name} if column_name else {},
                                             headers=headers)
                        if resp.status_code == 200: display_analysis_results(resp.json())
                        else: st.error("Analysis failed.")

        # Vertical Divider CSS Hook
        with divider_col:
            st.markdown("""
                <div style="border-left: 2px solid #86EFAC; height: 100%; min-height: 600px; margin: 0 auto;"></div>
            """, unsafe_allow_html=True)

        # Chatbot UI
        with chat_col:
            st.subheader("💬 Wellness Chat")
            chat_box = st.container(height=550)

            # 1. Render existing history first
            with chat_box:
                for turn in st.session_state.chat_history:
                    with st.chat_message(turn["role"]):
                        st.write(turn["content"])

            # 2. Get user input
            user_msg = st.chat_input("How are you feeling today?")

            if user_msg:
                # 3. Add to state and display user message instantly
                st.session_state.chat_history.append({"role": "user", "content": user_msg})
                with chat_box:
                    with st.chat_message("user"):
                        st.write(user_msg)

                    # 4. Display a spinner while waiting for the API response
                    with st.chat_message("assistant"):
                        with st.spinner("Thinking..."):
                            try:
                                resp = requests.post(
                                    f"{BACKEND_URL}/chat",
                                    json={"message": user_msg, "history": st.session_state.chat_history[-6:-1]},
                                    headers={"Authorization": f"Bearer {st.session_state.token}"},
                                    timeout=60
                                )
                                reply = resp.json()["reply"] if resp.status_code == 200 else "Service unavailable."
                            except:
                                reply = "Service unavailable."
                        st.write(reply)

                # 5. Append assistant reply to state
                st.session_state.chat_history.append({"role": "assistant", "content": reply})

# ---------------- 6. LOGOUT CLEANUP ----------------
elif choice == "🚪 Logout":
    st.session_state.token = None
    st.session_state.chat_history = []
    st.success("Session state arrays cleared.")
    st.rerun()

Overwriting app.py


## FastAPI backend (JWT-protected file upload)

This backend exposes a `/upload` endpoint that only accepts `.csv` or `.txt` files, verifies the same JWT issued at Streamlit login, and returns a preview (columns + first rows for CSV, first lines for TXT).


## Multilingual NLP pipeline module

Language detection, text cleaning, Telugu/Kannada stopword filtering, translation to English, lemmatization, VADER sentiment, and keyword-based emotion detection — imported by `backend.py`'s `/analyze` endpoint.


In [ ]:
%%writefile nlp_pipeline.py
import re
import ftfy
import emoji
import spacy
import torch
import stopwordsiso
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline as hf_pipeline
from langdetect import detect, DetectorFactory
from deep_translator import GoogleTranslator
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

DetectorFactory.seed = 0

_nlp = None
_vader = None
_qwen_model = None
_qwen_tokenizer = None
_bert_emotion_pipeline = None

QWEN_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
BERT_EMOTION_MODEL_NAME = "bhadresh-savani/bert-base-go-emotion"

LANGUAGE_NAMES = {
    "te": "Telugu", "kn": "Kannada", "en": "English", "ta": "Tamil",
    "hi": "Hindi", "ml": "Malayalam", "mr": "Marathi", "bn": "Bengali", "gu": "Gujarati",
}

def _get_stopwords(language_code: str) -> set:
    if stopwordsiso.has_lang(language_code):
        return stopwordsiso.stopwords(language_code)
    return set()

EMOTION_LABELS = ["Happy", "Sad", "Stress", "Angry", "Fear", "Neutral"]
EMOTION_EMOJI = {
    "Happy": "😊", "Sad": "😢", "Stress": "😫",
    "Angry": "😡", "Fear": "😨", "Neutral": "😐",
}

GOEMOTIONS_TO_APP_LABEL = {
    "joy": "Happy", "amusement": "Happy", "excitement": "Happy", "love": "Happy",
    "gratitude": "Happy", "optimism": "Happy", "relief": "Happy", "pride": "Happy",
    "admiration": "Happy", "approval": "Happy", "caring": "Happy", "sadness": "Sad",
    "disappointment": "Sad", "grief": "Sad", "remorse": "Sad", "nervousness": "Stress",
    "embarrassment": "Stress", "confusion": "Stress", "anger": "Angry", "annoyance": "Angry",
    "disgust": "Angry", "disapproval": "Angry", "fear": "Fear", "neutral": "Neutral",
    "realization": "Neutral", "surprise": "Neutral", "curiosity": "Neutral", "desire": "Neutral",
}

def _get_nlp():
    global _nlp
    if _nlp is None: _nlp = spacy.load("xx_sent_ud_sm")
    return _nlp

def _get_vader():
    global _vader
    if _vader is None: _vader = SentimentIntensityAnalyzer()
    return _vader

def _get_qwen():
    global _qwen_model, _qwen_tokenizer
    if _qwen_model is None:
        _qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_NAME)
        _qwen_model = AutoModelForCausalLM.from_pretrained(
            QWEN_MODEL_NAME,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto" if torch.cuda.is_available() else None,
        )
    return _qwen_model, _qwen_tokenizer

def _get_bert_emotion_pipeline():
    global _bert_emotion_pipeline
    if _bert_emotion_pipeline is None:
        _bert_emotion_pipeline = hf_pipeline(
            "text-classification", model=BERT_EMOTION_MODEL_NAME,
            top_k=None, device=0 if torch.cuda.is_available() else -1,
        )
    return _bert_emotion_pipeline

def _bert_emotion(text: str) -> dict:
    classifier = _get_bert_emotion_pipeline()
    if not text.strip(): text = "(empty feedback)"
    raw_predictions = classifier(text, truncation=True)[0]
    app_scores = {label: 0.0 for label in EMOTION_LABELS}
    for pred in raw_predictions:
        app_label = GOEMOTIONS_TO_APP_LABEL.get(pred["label"].lower(), "Neutral")
        app_scores[app_label] += pred["score"]
    total = sum(app_scores.values()) or 1.0
    app_scores = {label: round(score / total, 4) for label, score in app_scores.items()}
    final_emotion = max(app_scores, key=app_scores.get)
    return {"emotion": final_emotion, "scores": app_scores}

def process_employee_feedback(text: str) -> dict:
    nlp = _get_nlp()
    vader = _get_vader()
    raw_text = text  # Capture exactly for Milestone 2 output

    normalized_text = ftfy.fix_text(text)

    try: language = detect(normalized_text)
    except: language = "unknown"
    detected_language = LANGUAGE_NAMES.get(language, "Other / Unknown")

    emoji_list = [ch for ch in normalized_text if ch in emoji.EMOJI_DATA]

    cleaned_text = re.sub(r"https?://\S+|www\.\S+", " ", normalized_text)
    cleaned_text = re.sub(r"\S+@\S+", " ", cleaned_text)
    cleaned_text = re.sub(r"@\w+|#\w+", " ", cleaned_text)
    cleaned_text = emoji.replace_emoji(cleaned_text, replace="")
    cleaned_text = re.sub(r"\s+", " ", cleaned_text).strip()

    doc = nlp(cleaned_text)
    sentences = [s.text.strip() for s in doc.sents if s.text.strip()]
    original_tokens = [t.text for t in doc if not t.is_space]
    clean_tokens = [t.text for t in doc if not t.is_punct and not t.is_space and not t.like_num]

    selected_stopwords = _get_stopwords(language)
    filtered_tokens = [t for t in clean_tokens if t.lower() not in selected_stopwords]
    final_preprocessed_text = " ".join(filtered_tokens)

    try: translated_text = GoogleTranslator(source="auto", target="en").translate(final_preprocessed_text)
    except: translated_text = final_preprocessed_text

    english_doc = nlp(translated_text)
    # Extract specific lemmatized tokens for Milestone 2
    lemmatized_tokens = [t.lemma_ if t.lemma_ else t.text for t in english_doc if not t.is_space]
    lemmatized_text = " ".join(lemmatized_tokens)

    sentiment_scores = vader.polarity_scores(translated_text)
    compound_score = sentiment_scores["compound"]
    if compound_score >= 0.05: final_sentiment = "Positive 😊"
    elif compound_score <= -0.05: final_sentiment = "Negative 😔"
    else: final_sentiment = "Neutral 😐"

    bert_result = _bert_emotion(translated_text)
    final_emotion_label = bert_result["emotion"]
    final_emotion = f"{final_emotion_label} {EMOTION_EMOJI.get(final_emotion_label, '')}"

    return {
        "raw_text": raw_text,
        "language_code": language,
        "detected_language": detected_language,
        "cleaned_text": cleaned_text,
        "original_tokens": original_tokens,
        "filtered_tokens": filtered_tokens,
        "lemmatized_tokens": lemmatized_tokens,
        "emoji_list": emoji_list,
        "final_preprocessed_text": final_preprocessed_text,
        "translated_text": translated_text,
        "lemmatized_text": lemmatized_text,
        "sentiment_scores": sentiment_scores,
        "final_sentiment": final_sentiment,
        "emotion_scores": bert_result["scores"],
        "final_emotion": final_emotion,
        "sentences": sentences,
    }

CRISIS_KEYWORDS = ["suicide", "kill myself", "end my life", "want to die", "self harm", "self-harm", "hurt myself", "not worth living", "no reason to live"]
CRISIS_MESSAGE = "I'm really glad you reached out, and I want to make sure you get support beyond what I can offer here. If you're in immediate danger, please contact your local emergency number right now. You can also reach a crisis line: in India, AASRA is available at +91-9820466726 (24/7). If you're outside India, please look up a local crisis helpline or talk to a trusted person or your HR/EAP contact. You don't have to go through this alone."
WELLNESS_SYSTEM_PROMPT = "You are a supportive workplace wellness assistant for employees. Your role is to listen, validate feelings, and offer general, gentle coping suggestions. Keep replies short (2-4 sentences), warm, and non-judgmental. Avoid clinical labels and avoid being preachy or repetitive."

def _contains_crisis_language(text: str) -> bool:
    return any(kw in text.lower() for kw in CRISIS_KEYWORDS)

def wellness_chat_reply(message: str, history: list[dict] | None = None) -> dict:
    if _contains_crisis_language(message):
        return {"reply": CRISIS_MESSAGE, "flagged": True}
    model, tokenizer = _get_qwen()
    messages = [{"role": "system", "content": WELLNESS_SYSTEM_PROMPT}]
    for turn in (history or []):
        if turn.get("role") in ("user", "assistant") and turn.get("content"):
            messages.append({"role": turn["role"], "content": turn["content"]})
    messages.append({"role": "user", "content": message})
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=150, temperature=0.7, do_sample=True, top_p=0.9)
    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    reply = tokenizer.decode(generated, skip_special_tokens=True).strip()
    return {"reply": reply or "I'm here and listening — could you tell me a bit more about how you're feeling?", "flagged": False}

Writing nlp_pipeline.py


In [ ]:
%%writefile backend.py
import os, io, jwt, csv
from typing import Optional
from fastapi import FastAPI, UploadFile, File, Form, Header, HTTPException
from pydantic import BaseModel
from fastapi.middleware.cors import CORSMiddleware
from dotenv import load_dotenv
from nlp_pipeline import process_employee_feedback, wellness_chat_reply
load_dotenv()

SECRET = os.getenv("JWT_SECRET")
app = FastAPI(title="Upload API")

app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

def get_user(authorization: str = Header(None)):
    if not authorization or not authorization.startswith("Bearer "):
        raise HTTPException(401, "Missing token")
    token = authorization.split(" ", 1)[1]
    try: return jwt.decode(token, SECRET, algorithms=["HS256"])
    except jwt.PyJWTError: raise HTTPException(401, "Invalid or expired token")

def _extract_text_blob(raw: bytes, ext: str, column: str | None) -> tuple[str, str | None]:
    text = raw.decode("utf-8")
    if ext == "txt": return text.strip(), None
    reader = csv.reader(io.StringIO(text))
    rows = list(reader)
    if not rows or len(rows) == 1: raise HTTPException(400, "CSV file has no data rows.")
    header, data_rows = rows[0], rows[1:]
    col_index = header.index(column) if column and column in header else len(header) - 1
    values = [row[col_index] for row in data_rows if len(row) > col_index and row[col_index].strip()]
    blob = " ".join(values).strip()
    if not blob: raise HTTPException(400, "Chosen column has no text.")
    return blob, header[col_index]

@app.post("/analyze")
async def analyze(
    authorization: str = Header(None),
    file: Optional[UploadFile] = File(None),
    text_input: Optional[str] = Form(None),
    column: Optional[str] = Form(None)
):
    get_user(authorization)

    if text_input:
        text_blob = text_input
        used_column, name, ext = "Direct Input", "Direct Input", "TXT"
    elif file:
        name = file.filename or ""
        ext = name.lower().rsplit(".", 1)[-1] if "." in name else ""
        if ext not in ("csv", "txt"): raise HTTPException(400, "Only .csv or .txt allowed.")
        raw = await file.read()
        if len(raw) > 5 * 1024 * 1024: raise HTTPException(400, "File too large.")
        try: text_blob, used_column = _extract_text_blob(raw, ext, column)
        except UnicodeDecodeError: raise HTTPException(400, "Must be UTF-8 text.")
    else:
        raise HTTPException(400, "Provide either a file or text_input.")

    results = process_employee_feedback(text_blob)
    results.update({"filename": name, "file_type": ext.upper(), "used_column": used_column, "original_char_count": len(text_blob)})
    return results

@app.post("/upload")
async def upload(file: UploadFile = File(...), authorization: str = Header(None)):
    user = get_user(authorization)
    # ... keeping standard upload for preview purposes ...
    name = file.filename or ""
    ext = name.lower().rsplit(".", 1)[-1] if "." in name else ""
    raw = await file.read()
    text = raw.decode("utf-8")
    lines = text.splitlines()
    columns, preview_rows = None, None
    if ext == "csv":
        rows = list(csv.reader(io.StringIO(text)))
        if rows: columns, preview_rows = rows[0], rows[1:21]
    return {"filename": name, "type": ext, "uploaded_by": user["username"], "row_count": len(lines), "columns": columns, "preview_rows": preview_rows, "preview_lines": lines[:20] if ext != "csv" else None}

class ChatTurn(BaseModel): role: str; content: str
class ChatRequest(BaseModel): message: str; history: list[ChatTurn] = []

@app.post("/chat")
async def chat(payload: ChatRequest, authorization: str = Header(None)):
    get_user(authorization)
    if not payload.message.strip(): raise HTTPException(400, "Message empty.")
    return wellness_chat_reply(payload.message.strip(), history=[t.dict() for t in payload.history])

Writing backend.py


In [ ]:
from db import init_db
init_db()
print("✅ Connected to PostgreSQL and ensured tables exist.")

✅ Connected to PostgreSQL and ensured tables exist.


In [ ]:
from pyngrok import ngrok, conf
import subprocess, time

conf.get_default().auth_token = values["NGROK_AUTHTOKEN"]

# Kill any previous tunnels/streamlit/uvicorn instances from earlier runs in this session
ngrok.kill()
get_ipython().system_raw('pkill -f streamlit || true')
get_ipython().system_raw('pkill -f uvicorn || true')
time.sleep(1)

# Launch FastAPI (backend.py) in the background on port 8000 (internal only, not tunneled)
get_ipython().system_raw(
    'uvicorn backend:app --host 0.0.0.0 --port 8000 &'
)
time.sleep(5)  # NLP libs (spaCy model etc.) take a little longer to import

# Launch Streamlit in the background, quietly, on port 8501
get_ipython().system_raw(
    'streamlit run app.py --server.port 8501 --server.headless true '
    '--server.enableCORS false --server.enableXsrfProtection false &'
)
time.sleep(4)  # give both servers a moment to boot

public_url = ngrok.connect(8501, "http")
print(f"🚀 Your app is live at: {public_url}")
print("FastAPI backend is running internally on port 8000 (Streamlit talks to it via localhost).")
print("Open the URL above in your browser. Leave this Colab cell/runtime running to keep it up.")

🚀 Your app is live at: NgrokTunnel: "https://wick-flock-erased.ngrok-free.dev" -> "http://localhost:8501"
FastAPI backend is running internally on port 8000 (Streamlit talks to it via localhost).
Open the URL above in your browser. Leave this Colab cell/runtime running to keep it up.


In [ ]:
from pyngrok import ngrok
ngrok.kill()
get_ipython().system_raw('pkill -f streamlit || true')
get_ipython().system_raw('pkill -f uvicorn || true')
print("Stopped Streamlit, FastAPI, and closed ngrok tunnel.")

Stopped Streamlit, FastAPI, and closed ngrok tunnel.
